# JRN Plant List - Notes and validation

_Last updated: 2026-06-03_

This document has notes and validation checks for the new JRN plant list located at `jornada_im/dataprep/jrn520_taxa/plants/jrn_plant_list_MAIN.xlsx`. This version of the JRN plant list was derived from joining two source tables:

1. The most recent **EDI plant list**, (`JRN vascular plant species list.csv` or `jrn_plant_list_MAIN_YYYYMMDD.csv` in later versions) which itself came from a reformatted version of John Anderson’s list (Plntalfa_table_edit.xlsx) and Justin VanZee’s list (plantlistJER), and was maintained in Excel for a few years. This is a more inclusive list that includes over 500 taxa of the region. 
2. **John's plant list** that is maintained in the Field Crew Sharepoint directory (filename = `0_test-merge1.xlsx`). This is a less-inclusive list, focusing on taxa observed by the JRN field crew.

These two files were merged in R (see `reformat_main_plants.R` for details), and plant codes (LTER and USDA) and scientific names were then validated against the USDA Plants database (March 2026 version) and Kelly Allred's Flora of the Jornada Plain (9th edition) to produce a table of 573 taxa in the main list (`jrn_plants` in the spreadsheet) Plant traits mostly rely on **John's plant list**, but newer trait columns from the **EDI plant list** have been retained. Various synonyms have been extracted from both source lists and are collected in a new synonyms table that includes both out-of-date taxonomy, and accepted alternative taxonomies from USDA Plants or Allred when this list deviates from those for some reason. 

## Table descriptions

There are two related tables in the plant list. The main list (`plant_list` in the spreadsheet) and the list of synonyms (`plant_synonyms`). Descriptions for the two tables and their columns are below.

### The main plant list (`plant_list` sheet)

This is the main JRN plant list. It will be published as the next version of the **EDI plant list** when we're ready. There are 573 taxa, one row per taxon, sorted by `sciname`.

| Column | Derivation |
|--------|------------|
| `family` | From **John's plant list**, falling back to **EDI plant list** |
| `sciname` | Binomial name + infraspecific rank stripped of authority, derived by regex from `sciname_auth` |
| `sciname_auth` | Binomial name + infraspecific rank, with taxonomic authority. From **EDI plant list**, falling back to **John's plant list**, then USDA Plants; replaced with `sciname_auth_usda` (and old value moved to `alias`) where `sciname_usda_match` is FALSE, `sciname_auth_follows == "Other"`, and a USDA name is available |
| `usda_code` | From **John's plant list** or **EDI plant list** depending on manual review (see `use_john`/`use_main` vectors); resolved to accepted USDA symbol in most cases |
| `lter_code` | 4-character Jornada LTER species code; primary key; from **EDI plant list** (full join with **John's plant list**) |
| `common_name` | Common names from **John's plant list**, filled from USDA Plants then **Allred plant list**; new names appended as semicolon-delimited list, case-insensitive deduplication applied |
| `habit` | Categorical (`A`=annual, `B`=biennial, `P`=perennial); from **John's plant list**, falling back to **EDI plant list** |
| `form` | Categorical (`FERN`, `FORB`, `GRASS`, `LF-SU`, `S-SHR`, `SHRUB`, `ST-SU`, `TREE`, `VINE`); from **John's plant list**, falling back to **EDI plant list** |
| `cpath` | Categorical — photosynthetic pathway (`C3`, `C4`, `CAM`, `PAR`); from **John's plant list**, falling back to **EDI plant list** |
| `nativity` | Categorical (`native`, `introduced`); from **EDI plant list** |
| `habitat` | From **EDI plant list** |
| `phenology` | From **EDI plant list** |
| `reproduction` | Categorical (`seed`, `spore`); from **EDI plant list** |
| `lter_observed` | Categorical (`present`, `not observed`); from **EDI plant list** |
| `sciname_auth_follows` | Categorical — which source the accepted `sciname_auth` agrees with (`USDA Plants`, `Allred`, `Other`) |
| `usda_code_is_syn` | Boolean; `TRUE` if `usda_code` is a USDA synonym symbol rather than an accepted symbol |
| `note` | Free-text notes from **EDI plant list**, augmented during manual editing |

The last three columns are useful for diagnostic purposes. `usda_code_is_syn` indicates which of the codes in `usda_code` is a synonym in the USDA Plants database. Taxonomy for these species has probably been updated and there is a new accepted code for the taxa. This will be listed in the `synonyms` table. `sciname_auth_follows` indicates when the `sciname_auth` column, ultimately derived from the LTER taxonomy in the previous **EDI plant list**, matches the USDA Plants taxonomy associated with that USDA code, Kelly Allred's taxonomy, or something else. Where "Other", the taxon name may be following a preferred local authority, outdated taxonomy (old USDA taxa), or there may be a subtle difference between `sciname_auth` and the same taxa in USDA or Allred taxonomy (such as author abbreviations or infrapecific rank delimiter). Also note that most of the time, USDA plants and Allred agree, they just write scientific names and authorities slightly differently. The `notes` column collects notes about conflicts between taxonomic authorities, duplicate codes, and other issues.

---

### Synonyms (`plant_synonyms` sheet)

This table contains collected synonyms for the taxa in the JRN plant list and will be published alongside the new plant list. The values in the `usda_code` and `lter_code` columns are drawn from the main plant list (foreign keys to the same columns in `plant_list`). There is one row per synonym, and there may be 0-to-many synonyms per taxon in the main list. The table is de-duplicated but there are still some near-duplicate synonyms that can be removed after manual review. Sorted by `lter_code`.

| Column | Derivation |
|--------|------------|
| `usda_code` | Accepted USDA code for the taxon (from `plant_list`) |
| `lter_code` | LTER code for the taxon (from `plant_list`) |
| `sciname_auth` | The synonym scientific name (one alias entry per row, split from the semicolon-delimited `alias` column; or an old `Direct USDA Code` entry from **John's plant list**) |
| `usda_code_syn` | USDA symbol corresponding to the synonym name: matched by sciname against USDA synonym rows; when `usda_code_is_syn` is TRUE, falls back to matching against USDA accepted rows and taking `usda_code`; `NA` if no match found |


In [82]:
# Load libraries and configs
library(tidyverse)
library(readxl)
readRenviron("../../.Renviron")
source(file.path("../../config.R"))

In [83]:
# Path to the plant list files (jrn520)
plants_path <- paste(im_path, "dataprep", "jrn520_taxa", "plants", sep="/")
# Load the plant list and synonyms
mainpl <- read_excel(file.path(plants_path, "jrn_plant_list_MAIN.xlsx"), sheet='plant_list',
                        skip=4, na = c(".", "NA"))
syn <- read_excel(file.path(plants_path, "jrn_plant_list_MAIN.xlsx"), sheet='plant_synonyms',
                        skip=4, na = c(".", "NA"))

## LTER and USDA code validation checks

Ideally, all taxa included in the main JRN plant list should have one unique LTER code and a unique USDA Plants code. This isn't always the case:

* The same LTER code may be used to refer to two unique taxa (duplicate LTER codes)
* Multiple LTER codes may refer to the same taxon (synonym LTER codes, duplicate USDA codes)
* The same USDA code may refer to the same taxon, but multiple LTER codes, indicating the LTER codes are synonyms (converse of the above).
* Some taxa are missing an LTER code.

Summary data for these cases are below, and cases are examined in depth after.

In [84]:
# Check: Are there duplicate lter codes?
lter_code_dup <- mainpl |>
  filter(!is.na(lter_code)) |>
  group_by(lter_code) |> filter(n() > 1) |>
  ungroup() |> arrange(lter_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of duplicate lter_code values in mainpl:", nrow(lter_code_dup)/2, "\n")

# Check: Duplicate LTER codes with multiple USDA codes
lter_multi_usda <- mainpl |>
  filter(!is.na(lter_code), !is.na(usda_code)) |>
  group_by(lter_code) |> filter(n_distinct(usda_code) > 1) |>
  ungroup() |> arrange(lter_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of lter_code values with multiple usda_codes:", n_distinct(lter_multi_usda$lter_code), "\n")
#if (nrow(lter_multi_usda) > 0) print(lter_multi_usda, n=50)

# Check: Are there duplicate usda codes?
usda_code_dup <- mainpl |>
  filter(!is.na(usda_code)) |>
  group_by(usda_code) |> filter(n() > 1) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of duplicate usda_code values in mainpl:", nrow(usda_code_dup)/2, "\n")

# Check: Duplicate USDA codes with multiple LTER codes
usda_multi_lter <- mainpl |>
  filter(!is.na(usda_code), !is.na(lter_code)) |>
  group_by(usda_code) |> filter(n_distinct(lter_code) > 1) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of duplicate usda_code values with multiple lter_codes:", n_distinct(usda_multi_lter$usda_code), "\n")

# Check: Missing LTER codes
lter_na <- mainpl |>
  filter(is.na(lter_code)) |>
  ungroup() |> arrange(usda_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of missing lter_codes:", nrow(lter_na), "\n")

# Check: Missing USDA codes
usda_na <- mainpl |>
  filter(is.na(usda_code)) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of missing usda_codes:", nrow(usda_na), "\n")

Number of duplicate lter_code values in mainpl: 11 
Number of lter_code values with multiple usda_codes: 10 
Number of duplicate usda_code values in mainpl: 7 
Number of duplicate usda_code values with multiple lter_codes: 6 
Number of missing lter_codes: 5 
Number of missing usda_codes: 0 



### First check duplicate LTER codes

Below is a table of all duplicate LTER codes in the list.

In [85]:
if (nrow(lter_code_dup) > 0) print(lter_code_dup, n=50)

# A tibble: 22 × 3
   lter_code usda_code sciname_auth                                                                 
   <chr>     <chr>     <chr>                                                                        
 1 ACCO      ACCOC     Acacia constricta Benth. var. constricta                                     
 2 ACCO      ACCOP9    Acacia constricta var. paucispina Woot. & Standl.                            
 3 BOCC      BOCO2     Boerhavia coulteri (Hooker f.) S. Watson var. coulteri                       
 4 BOCC      BOCO2     Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S. Watson) Spellenberg
 5 BOCU      BOCUC     Bouteloua curtipendula (Michx.) Torr. var. caespitosa Gould & Kapadia        
 6 BOCU      BOCUC2    Bouteloua curtipendula (Michx.) Torr. var. curtipendula                      
 7 CHLI      CHLIA     Chilopsis linearis (Cav.) Sweet. subsp. arcuata Fosberg                      
 8 CHLI      CHLIL2    Chilopsis linearis (Cav.) Sweet. subsp. linearis 

Of these duplicate LTER codes, all refer to multiple taxonomic entities - either different species or different infraspecific taxa - that each have unique USDA codes. If these are officially recognized and observed taxa on the Jornada, then in some cases we may want to assign a unique LTER code for each taxon. In a few cases the duplicate LTER codes are more a product of discrepancies between the taxonomic authorities that were used to compose the list. 

In the case of `TECO` and `MESC`, it appears that one authority recognizes a subspecies and one doesn't. 

In [86]:
lter_code_dup[lter_code_dup$lter_code %in% c('TECO', 'MESC'),]

# A tibble: 4 × 3
  lter_code usda_code sciname_auth                                                                 
  <chr>     <chr>     <chr>                                                                        
1 MESC      MESC      Menodora scabra Gray                                                         
2 MESC      MESCL     Menodora scabra Gray var. laevis (Woot. & Standl.) Steyerm.                  
3 TECO      TECO      Tetraclea coulteri Gray.                                                     
4 TECO      TECOA     Tetraclea coulteri Gray var. angustifolia (Woot. & Standl.) Nelson & Macbride


Another exception is `BOCC`, which refers to two taxa (var. couteri and var. palmeri), but these have only one USDA code. This is because the two infraspecific taxa the codes refer to are not recognized in USDA Plants.

In [87]:
lter_code_dup[!(lter_code_dup$lter_code %in% lter_multi_usda$lter_code),]

# A tibble: 2 × 3
  lter_code usda_code sciname_auth                                                                 
  <chr>     <chr>     <chr>                                                                        
1 BOCC      BOCO2     Boerhavia coulteri (Hooker f.) S. Watson var. coulteri                       
2 BOCC      BOCO2     Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S. Watson) Spellenberg

### Now check duplicate USDA codes

Below is a table of duplicate USDA codes, which can help identify when multiple LTER codes are synonyms for the same taxa.

In [88]:
if (nrow(usda_code_dup) > 0) print(usda_code_dup)

# A tibble: 14 × 3
   usda_code lter_code sciname_auth                                                                   
   <chr>     <chr>     <chr>                                                                          
 1 ARPUN     ARNE      Aristida purpurea Nutt. var. nealleyi (Vasey) Allred                           
 2 ARPUN     ARGL      Aristida purpurea Nutt. var. nealleyi (Vasey) Allred                           
 3 BOCO2     BOCC      Boerhavia coulteri (Hooker f.) S. Watson var. coulteri                         
 4 BOCO2     BOCC      Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S. Watson) Spellenberg  
 5 DAPU7     DAPU      Dasyochloa pulchella (HBK) Hitchc.                                             
 6 DAPU7     ERPU      Dasyochloa pulchella (Kunth) Willd. ex Rydb.                                   
 7 ERPEM     ERMI      Eragrostis pectinacea (Michx.) Nees var. miserrima (Fourn.) J.Reeder           
 8 ERPEM     ERAR      Eragrostis pectinacea (Michx.) 

In almost all cases here, the duplicate USDA code refers to two synonym LTER codes. Here it would make sense to pick one LTER code and retire the other. The exception is `BOCO2`, which is the closest USDA Plants match to the two taxa that the `BOCC` LTER code refers to. Also note some discrepancies in how taxonomic authorities are listed in the scientific names, such as for USDA code `ERPEM`. This probably indicates slightly different authorities used in the plant lists that were the source for this dataset (i.e. John's list versus the EDI list).

### Taxa with missing LTER codes

In [89]:
if (nrow(lter_na) > 0) print(lter_na)

# A tibble: 5 × 3
  lter_code usda_code sciname_auth                                                    
  <chr>     <chr>     <chr>                                                           
1 NA        ASSIS     Astrolepis sinuata (Lag. ex Sw.) Benham & Windham subsp. sinuata
2 NA        BRAR71    Bryum argenteum Hedw.                                           
3 NA        CHLI      Cheilanthes lindheimeri Hook.                                   
4 NA        DECO10    Desmatodon convolutus (Brid.) Grout                             
5 NA        WECO2     Weissia condensa (Voit) Lindb.                                  


All of these are spore plants where the standard LTER code (first 2 letters of genus & species) would cause a conflict with existing LTER codes.

## Scientific name and authority validation checks

Now lets look at whether the scientific name and authority (`sciname_auth` column) for each taxon matches the taxonomy in USDA Plants or not.

In [90]:
# Load USDA plants list and create lookup tables
usda <- read_csv(file.path(plants_path, 'usda_plantlst_20260318.txt')) |>
  rename_with(tolower) |>
  rename(usda_code = symbol, usda_code_syn = `synonym symbol`,
         sciname_auth = `scientific name with author`) |>
  mutate(sciname_auth = str_replace(sciname_auth, " ssp. ", " subsp. "))

# Helper to strip authority from sciname_auth — reused in both lookups below
derive_sciname <- function(x) {
  base  <- str_extract(x, "^[A-Z][a-z-]+ [a-z×-]+")
  infra <- str_extract(x, "(?<= )(?:subsp\\.|ssp\\.|var\\.|f\\.|forma) [a-z-]+")
  if_else(is.na(infra), base, paste(base, infra))
}

# Lookup 1: USDA synonym rows — sciname -> usda_code_syn
usda_syn_lookup <- usda |>
  filter(!is.na(usda_code_syn)) |>
  mutate(sciname_syn = derive_sciname(sciname_auth)) |>
  select(usda_code_syn, sciname_syn, sciname_auth)

# Lookup 2: USDA accepted rows — sciname -> usda_code (used when usda_code_is_syn is TRUE)
usda_accepted_lookup <- usda |>
  filter(is.na(usda_code_syn)) |>
  mutate(sciname_syn = derive_sciname(sciname_auth)) |>
  select(usda_code_accepted = usda_code, sciname_syn, sciname_auth)

Rows: 93157 Columns: 5
── Column specification ───────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (5): Symbol, Synonym Symbol, Scientific Name with Author, Common Name, Family

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [91]:
# Check: exact sciname_auth match against USDA Plants
# Accepted taxa: compare against accepted sciname_auth via usda_accepted_lookup
# Synonym taxa: compare against synonym row sciname_auth via usda_syn_lookup
sciname_auth_check <- bind_rows(
  mainpl |>
    filter(!usda_code_is_syn) |>
    left_join(usda_accepted_lookup |> select(usda_code_accepted, sciname_auth_usda = sciname_auth),
              by = c("usda_code" = "usda_code_accepted")),
  mainpl |>
    filter(usda_code_is_syn) |>
    left_join(usda_syn_lookup |> select(usda_code_syn, sciname_auth_usda = sciname_auth),
              by = c("usda_code" = "usda_code_syn"))
) |>
  mutate(sciname_auth_usda_match = sciname_auth == sciname_auth_usda)

cat("Number of taxa where sciname_auth matches USDA Plants exactly:", sum(sciname_auth_check$sciname_auth_usda_match, na.rm = TRUE), "\n")
cat("Number of taxa where sciname_auth does not match USDA Plants:", sum(!sciname_auth_check$sciname_auth_usda_match, na.rm = TRUE), "\n")
cat("Number of taxa where sciname_auth could not be checked (no USDA entry found):", sum(is.na(sciname_auth_check$sciname_auth_usda_match)), "\n")

Number of taxa where sciname_auth matches USDA Plants exactly: 405 
Number of taxa where sciname_auth does not match USDA Plants: 168 
Number of taxa where sciname_auth could not be checked (no USDA entry found): 0 


## Other notes from manual review

**Uncertain infraspecific taxa IDs**

* _Astragalus nutallianus var. austrinus_ (USDA code ASNUA, LTER code ASNU) was present in John's list, but Allred does not confirm the infraspecific.
* _Astragalus wootonii_ (USDA code ASWO2, LTER code ASWO) is called _Astragalus allochrous var. playanus_ (ASALP USDA code) by Allred.
* _Menodora scabra var. scabra_ was listed as the infraspecific taxa in the EDI list but I can't find it anywhere (USDA Plants or Allred)

**Other issues and notes**

* TAAU and TAAN (& their corresponding USDA codes) have both been subsumed into PHAU13 in USDA Plants
* BOTO2 has been subsumed into BOSP in USDA plants
* VEAM (Verbena ambrosifolia) in the EDI list may be out-of-date. Allred has two taxa, Glandularia pubera (GLBIB2) and G. wrightii (GLBIC), but pubera seems to be subsumed into GLBIC according to USDA Plants.
* Similarly, PORE is probably out of date and has been subsumed into POOL.
* ECTR LTER code occurrs twice. According to Allred "Echinocereus triglochidiatus Engelmann var. gurneyi" (ECTRG2) has been misapplied and ECTR should just refer to ECCOC.

Its worth noting that Allred very often is unaware of, or just not using, an up-to-date USDA code, especially for infraspecific rank taxa. There are also some typos in USDA codes and binomial names. Some of these are corrected in v4 of Greg's Allred table. 

Also note that **John's plant list** and the **EDI plant list** fairly often have infraspecific taxa that are not matched to the USDA code they use. Often these infraspecific taxa are not found in Allred. Example: LTER code ASTE.

In [92]:
system("jupyter nbconvert './plant_list_notes_and_val.ipynb' --no-input --to html")

[NbConvertApp] Converting notebook ./plant_list_notes_and_val.ipynb to html
[NbConvertApp] Writing 301079 bytes to plant_list_notes_and_val.html
